In [3]:
import numpy as np

X = np.array([2.0, 4.0])
y = 6.0
lr = 0.01
EPOCHS = 200

W1 = np.array([[0.21, 0.77],
               [0.24, 0.56]])
B1 = np.array([0.71, 0.45])
W2 = np.array([[0.98, 0.65],
               [0.31, 0.52]])
B2 = np.array([0.88, 0.22])
W3 = np.array([0.48, 0.17])
B3 = 0.51


def relu(z):
    return np.maximum(0.0, z)


def relu_grad(z):
    return (z > 0).astype(float)


def forward(x, W1, B1, W2, B2, W3, B3):
    Z1 = W1 @ x + B1
    A1 = relu(Z1)
    Z2 = W2 @ A1 + B2
    A2 = relu(Z2)
    Z3 = W3 @ A2 + B3
    return Z1, A1, Z2, A2, Z3


def backward(x, target, W1, B1, W2, B2, W3, B3):
    Z1, A1, Z2, A2, Z3 = forward(x, W1, B1, W2, B2, W3, B3)
    y_hat = Z3
    J = 0.5 * (y_hat - target) ** 2

    dJ_dyhat = y_hat - target
    dJ_dZ3 = dJ_dyhat

    dW3 = dJ_dZ3 * A2
    dB3 = dJ_dZ3

    dJ_dA2 = dJ_dZ3 * W3
    dJ_dZ2 = dJ_dA2 * relu_grad(Z2)
    dW2 = np.outer(dJ_dZ2, A1)
    dB2 = dJ_dZ2

    dJ_dA1 = W2.T @ dJ_dZ2
    dJ_dZ1 = dJ_dA1 * relu_grad(Z1)
    dW1 = np.outer(dJ_dZ1, x)
    dB1 = dJ_dZ1

    cache = dict(Z1=Z1, A1=A1, Z2=Z2, A2=A2, Z3=Z3, y_hat=y_hat, J=J,
                 dJ_dyhat=dJ_dyhat, dJ_dA2=dJ_dA2, dJ_dZ2=dJ_dZ2,
                 dJ_dA1=dJ_dA1, dJ_dZ1=dJ_dZ1)
    grads = dict(W1=dW1, B1=dB1, W2=dW2, B2=dB2, W3=dW3, B3=dB3)
    return cache, grads


def names_values(W1, B1, W2, B2, W3, B3):
    return [
        ("W11", W1[0, 0]), ("W12", W1[0, 1]), ("W13", W1[1, 0]), ("W14", W1[1, 1]),
        ("B11", B1[0]), ("B12", B1[1]),
        ("W21", W2[0, 0]), ("W22", W2[0, 1]), ("W23", W2[1, 0]), ("W24", W2[1, 1]),
        ("B21", B2[0]), ("B22", B2[1]),
        ("W31", W3[0]), ("W32", W3[1]),
        ("B3", B3),
    ]


def grad_list(g):
    return [
        g["W1"][0, 0], g["W1"][0, 1], g["W1"][1, 0], g["W1"][1, 1],
        g["B1"][0], g["B1"][1],
        g["W2"][0, 0], g["W2"][0, 1], g["W2"][1, 0], g["W2"][1, 1],
        g["B2"][0], g["B2"][1],
        g["W3"][0], g["W3"][1],
        g["B3"],
    ]


def sgd_step(params, grads, lr):
    W1, B1, W2, B2, W3, B3 = params
    return (W1 - lr * grads["W1"], B1 - lr * grads["B1"],
            W2 - lr * grads["W2"], B2 - lr * grads["B2"],
            W3 - lr * grads["W3"], B3 - lr * grads["B3"])


print("\n1. Initial parameters")
for n, v in names_values(W1, B1, W2, B2, W3, B3):
    print(f"  {n} = {v:.6f}")

cache, grads = backward(X, y, W1, B1, W2, B2, W3, B3)

print("\n2. Forward propagation")
print(f"  Z11 = {cache['Z1'][0]:.6f}   A11 = {cache['A1'][0]:.6f}")
print(f"  Z12 = {cache['Z1'][1]:.6f}   A12 = {cache['A1'][1]:.6f}")
print(f"  Z21 = {cache['Z2'][0]:.6f}   A21 = {cache['A2'][0]:.6f}")
print(f"  Z22 = {cache['Z2'][1]:.6f}   A22 = {cache['A2'][1]:.6f}")
print(f"  Z3 = y_hat = {cache['y_hat']:.6f}")

print("\n3. Loss")
print(f"  J = 0.5*(y_hat - y)^2 = 0.5*({cache['y_hat']:.6f} - {y})^2 = {cache['J']:.6f}")

print("\n4. Backpropagation")
print(f"  dJ/dy_hat = {cache['dJ_dyhat']:.6f}")
print(f"  dJ/dA21 = {cache['dJ_dA2'][0]:.6f}   dJ/dA22 = {cache['dJ_dA2'][1]:.6f}")
print(f"  dJ/dZ21 = {cache['dJ_dZ2'][0]:.6f}   dJ/dZ22 = {cache['dJ_dZ2'][1]:.6f}")
print(f"  dJ/dA11 = {cache['dJ_dA1'][0]:.8f}   dJ/dA12 = {cache['dJ_dA1'][1]:.8f}")
print(f"  dJ/dZ11 = {cache['dJ_dZ1'][0]:.8f}   dJ/dZ12 = {cache['dJ_dZ1'][1]:.8f}")

print("\n5. All gradients")
for (n, _), g in zip(names_values(W1, B1, W2, B2, W3, B3), grad_list(grads)):
    print(f"  dJ/d{n} = {g:.8f}")

new_params = sgd_step((W1, B1, W2, B2, W3, B3), grads, lr)

print(f"\n6. Parameter updates (lr = {lr})")
old_vals = names_values(W1, B1, W2, B2, W3, B3)
new_vals = names_values(*new_params)
for (n, old), (_, new), g in zip(old_vals, new_vals, grad_list(grads)):
    print(f"  {n}_new = {old:.6f} - {lr}*({g:.8f}) = {new:.8f}")

*_, y_hat_after = forward(X, *new_params)
J_after = 0.5 * (y_hat_after - y) ** 2
print("\n7. After the first step")
print(f"  y_hat = {y_hat_after:.6f}   J = {J_after:.6f}")

params = (W1, B1, W2, B2, W3, B3)
for epoch in range(EPOCHS):
    _, g = backward(X, y, *params)
    params = sgd_step(params, g, lr)

print("\n" + line)
print(f"FINAL RESULT AFTER {EPOCHS} EPOCHS")
print(line)

*_, y_final = forward(X, *params)
J_final = 0.5 * (y_final - y) ** 2
for n, v in names_values(*params):
    print(f"  {n} = {v:.6f}")
print(f"\n  y_hat = {y_final:.6f}   (target y = {y})")
print(f"  J = {J_final:.10f}")


1. Initial parameters
  W11 = 0.210000
  W12 = 0.770000
  W13 = 0.240000
  W14 = 0.560000
  B11 = 0.710000
  B12 = 0.450000
  W21 = 0.980000
  W22 = 0.650000
  W23 = 0.310000
  W24 = 0.520000
  B21 = 0.880000
  B22 = 0.220000
  W31 = 0.480000
  W32 = 0.170000
  B3 = 0.510000

2. Forward propagation
  Z11 = 4.210000   A11 = 4.210000
  Z12 = 3.170000   A12 = 3.170000
  Z21 = 7.066300   A21 = 7.066300
  Z22 = 3.173500   A22 = 3.173500
  Z3 = y_hat = 4.441319

3. Loss
  J = 0.5*(y_hat - y)^2 = 0.5*(4.441319 - 6.0)^2 = 1.214743

4. Backpropagation
  dJ/dy_hat = -1.558681
  dJ/dA21 = -0.748167   dJ/dA22 = -0.264976
  dJ/dZ21 = -0.748167   dJ/dZ22 = -0.264976
  dJ/dA11 = -0.81534603   dJ/dA12 = -0.62409587
  dJ/dZ11 = -0.81534603   dJ/dZ12 = -0.62409587

5. All gradients
  dJ/dW11 = -1.63069206
  dJ/dW12 = -3.26138412
  dJ/dW13 = -1.24819174
  dJ/dW14 = -2.49638349
  dJ/dB11 = -0.81534603
  dJ/dB12 = -0.62409587
  dJ/dW21 = -3.14978256
  dJ/dW22 = -2.37168901
  dJ/dW23 = -1.11554799
  dJ/dW2

In [4]:
"""
Predictive modeling with machine learning (simple version).
Task: classify a breast tumour as malignant or benign (Breast Cancer Wisconsin dataset).
Models: Logistic Regression, Decision Tree, Neural Network (MLP).
"""
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
OUT = "work2"
os.makedirs(OUT, exist_ok=True)

# 1. Data ---------------------------------------------------------------
data = load_breast_cancer(as_frame=True)
X, y = data.data.copy(), data.target.copy()   # 0 = malignant, 1 = benign
print("Shape:", X.shape, "| classes:", y.value_counts().to_dict())
print("Missing:", int(X.isna().sum().sum()), "| duplicates:", int(X.duplicated().sum()))

# 2. Feature engineering (row-wise, so no leakage) ------------------------
X["area_to_perimeter"] = X["mean area"] / X["mean perimeter"]
X["worst_to_mean_radius"] = X["worst radius"] / X["mean radius"]

# 3. Split, then outliers: clip by IQR bounds computed on TRAIN only -------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

q1, q3 = X_train.quantile(0.25), X_train.quantile(0.75)
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
outlier_share = float(((X_train < lo) | (X_train > hi)).values.mean())
print(f"Outlier cells in train (IQR rule): {outlier_share:.2%}")
X_train = X_train.clip(lo, hi, axis=1)
X_test = X_test.clip(lo, hi, axis=1)

# 4. Models (scaling is inside the pipeline) ------------------------------
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "Decision Tree": make_pipeline(DecisionTreeClassifier(max_depth=4, random_state=SEED)),
    "Neural Network (MLP)": make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=SEED)),
}

# 5. 5-fold stratified cross-validation on the train set ------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for m in scoring:
        row[f"{m}_mean"] = res[f"test_{m}"].mean()
        row[f"{m}_std"] = res[f"test_{m}"].std()
    cv_rows.append(row)
cv_df = pd.DataFrame(cv_rows).set_index("model").round(4)
print("\n=== CV (mean) ===")
print(cv_df[[f"{m}_mean" for m in scoring]])

# 6. Hold-out test ----------------------------------------------------------
test_rows, rocs, cms = [], {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba)})
    rocs[name] = roc_curve(y_test, proba)[:2]
    cms[name] = confusion_matrix(y_test, pred).tolist()
test_df = pd.DataFrame(test_rows).set_index("model").round(4)
print("\n=== Test ===")
print(test_df)
print("\nConfusion matrices [[TN, FP],[FN, TP]] (0=malignant, 1=benign):", cms)

# 7. Plots -----------------------------------------------------------------
plt.figure(figsize=(6, 5))
for name, (fpr, tpr) in rocs.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC={test_df.loc[name, 'roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves (test set)"); plt.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/roc_curves.png", dpi=150); plt.close()

coef = pd.Series(models["Logistic Regression"][-1].coef_[0], index=X.columns)
top = coef.reindex(coef.abs().sort_values().index)[-10:]
plt.figure(figsize=(7, 4.5)); top.plot.barh()
plt.title("Logistic Regression: top-10 coefficients (standardized features)")
plt.tight_layout(); plt.savefig(f"{OUT}/coefficients.png", dpi=150); plt.close()

tree = models["Decision Tree"][-1]
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)[:5]
print("\nTree top features:", imp.round(4).to_dict())
print("LR top coefficients:", top.sort_values(key=abs, ascending=False).round(3).head(5).to_dict())

cv_df.to_csv(f"{OUT}/cv_results.csv"); test_df.to_csv(f"{OUT}/test_results.csv")
json.dump({"outlier_share_train": outlier_share, "confusion_matrices": cms,
           "train_size": len(X_train), "test_size": len(X_test)},
          open(f"{OUT}/summary.json", "w"), indent=2)

# 8. Save the best model (by test ROC-AUC) for deployment -------------------
import joblib
best = test_df["roc_auc"].idxmax()
joblib.dump(models[best], f"{OUT}/model.joblib")
print("Saved model:", best)

Shape: (569, 30) | classes: {1: 357, 0: 212}
Missing: 0 | duplicates: 0
Outlier cells in train (IQR rule): 3.90%

=== CV (mean) ===
                      accuracy_mean  precision_mean  recall_mean  f1_mean  \
model                                                                       
Logistic Regression          0.9648          0.9659       0.9789   0.9721   
Decision Tree                0.9253          0.9407       0.9404   0.9403   
Neural Network (MLP)         0.9758          0.9760       0.9860   0.9808   

                      roc_auc_mean  
model                               
Logistic Regression         0.9960  
Decision Tree               0.9151  
Neural Network (MLP)        0.9965  

=== Test ===
                      accuracy  precision  recall      f1  roc_auc
model                                                             
Logistic Regression     0.9825     0.9861  0.9861  0.9861   0.9977
Decision Tree           0.9211     0.9701  0.9028  0.9353   0.9456
Neural Network 

In [6]:
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
OUT = "output"
os.makedirs(OUT, exist_ok=True)
URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
LOCAL = "telco.csv"

df = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)
print("Shape:", df.shape)
print("Churn:", df["Churn"].value_counts().to_dict(),
      f"({(df['Churn'] == 'Yes').mean():.2%} churn)")
print("Duplicate rows:", int(df.duplicated().sum()))

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_missing = int(df["TotalCharges"].isna().sum())
print("Missing TotalCharges:", n_missing,
      "| their tenure:", df.loc[df["TotalCharges"].isna(), "tenure"].unique().tolist())
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    print(f"IQR outliers in {c}: {n_out}")

addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]
df["n_addons"] = (df[addons] == "Yes").sum(axis=1)
df["avg_monthly_charge"] = df["TotalCharges"] / df["tenure"].clip(lower=1)

y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])

num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges",
            "n_addons", "avg_monthly_charge"]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

def prep():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

models = {
    "Logistic Regression": Pipeline([("prep", prep()),
        ("clf", LogisticRegression(max_iter=2000))]),
    "Decision Tree": Pipeline([("prep", prep()),
        ("clf", DecisionTreeClassifier(max_depth=5, random_state=SEED))]),
    "Neural Network (MLP)": Pipeline([("prep", prep()),
        ("clf", MLPClassifier(hidden_layer_sizes=(32,), early_stopping=True,
                              max_iter=500, random_state=SEED))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for m in scoring:
        row[f"{m}_mean"] = res[f"test_{m}"].mean()
        row[f"{m}_std"] = res[f"test_{m}"].std()
    cv_rows.append(row)
cv_df = pd.DataFrame(cv_rows).set_index("model").round(4)
print("\n=== 5-fold CV on train (mean) ===")
print(cv_df[[f"{m}_mean" for m in scoring]].to_string())
print("(std)")
print(cv_df[[f"{m}_std" for m in scoring]].to_string())

test_rows, rocs, cms = [], {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba)})
    rocs[name] = roc_curve(y_test, proba)[:2]
    cms[name] = confusion_matrix(y_test, pred).tolist()
test_df = pd.DataFrame(test_rows).set_index("model").round(4)
print("\n=== Test ===")
print(test_df.to_string())
print("\nConfusion matrices [[TN, FP],[FN, TP]] (1 = churn):")
for k, v in cms.items():
    print(" ", k, v)
base = float(max(y_test.mean(), 1 - y_test.mean()))
print(f"\nBaseline accuracy (always 'no churn'): {base:.4f}")

lr = models["Logistic Regression"]
names = lr["prep"].get_feature_names_out()
coef = pd.Series(lr["clf"].coef_[0], index=names)
top = coef.reindex(coef.abs().sort_values(ascending=False).index)[:8]
print("\nLR top coefficients:", top.round(3).to_dict())
tree = models["Decision Tree"]
imp = pd.Series(tree["clf"].feature_importances_, index=names).sort_values(ascending=False)[:5]
print("Tree top features:", imp.round(4).to_dict())

plt.figure(figsize=(6, 5))
for name, (fpr, tpr) in rocs.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC={test_df.loc[name, 'roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves (test set)"); plt.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/roc_curves.png", dpi=150); plt.close()

plt.figure(figsize=(7.5, 4.5))
top.sort_values().plot.barh()
plt.title("Logistic Regression: largest coefficients")
plt.tight_layout(); plt.savefig(f"{OUT}/coefficients.png", dpi=150); plt.close()

cv_df.to_csv(f"{OUT}/cv_results.csv"); test_df.to_csv(f"{OUT}/test_results.csv")
json.dump({"n_train": len(X_train), "n_test": len(X_test), "n_missing_total_charges": n_missing,
           "n_features_after_encoding": int(len(names)), "baseline_accuracy": base,
           "confusion_matrices": cms}, open(f"{OUT}/summary.json", "w"), indent=2)

import joblib
best = test_df["roc_auc"].idxmax()
joblib.dump(models[best], f"{OUT}/model.joblib")
print("Saved model:", best)

Shape: (7043, 21)
Churn: {'No': 5174, 'Yes': 1869} (26.54% churn)
Duplicate rows: 0
Missing TotalCharges: 11 | their tenure: [0]
IQR outliers in tenure: 0
IQR outliers in MonthlyCharges: 0
IQR outliers in TotalCharges: 0

=== 5-fold CV on train (mean) ===
                      accuracy_mean  precision_mean  recall_mean  f1_mean  roc_auc_mean
model                                                                                  
Logistic Regression          0.8025          0.6535       0.5445   0.5932        0.8457
Decision Tree                0.7934          0.6364       0.5284   0.5741        0.8293
Neural Network (MLP)         0.8017          0.6640       0.5130   0.5779        0.8422
(std)
                      accuracy_std  precision_std  recall_std  f1_std  roc_auc_std
model                                                                             
Logistic Regression         0.0114         0.0253      0.0437  0.0302       0.0126
Decision Tree               0.0079         0.0295